In [1]:
#!/usr/bin/env sage -python
# -*- coding: utf-8 -*-

import os
import sys
import csv
import math
import argparse
from collections import OrderedDict

from sage.all import RR, sqrt, log, oo


NOTEBOOK_DIR = "/home/ubuntu/桌面/AsiaCrypt/LatticeEstimator/notebook"
PROJECT_DIR = "/home/ubuntu/桌面/AsiaCrypt/LatticeEstimator"

DEFAULT_ESTIMATOR_PATH = os.environ.get(
    "LATTICE_ESTIMATOR_PATH",
    PROJECT_DIR,
)


DEFAULT_PROFILES = OrderedDict([
    # name: target, q, n, k, eta_s, eta_r, t_pk, t_u, t_v, b_msg, v_count, ss_bits
    #
    # Length convention in this script:
    #   FO message bits = shared secret bits = target security bits.
    #   z bits          = shared secret bits = target security bits.
    #
    # b_msg and v_count still describe the scalar/E8 codec capacity.
    # For n = 256:
    #   b_msg = 1, v_count = 1 gives 256 bits of scalar capacity.
    #   b_msg = 2, v_count = 1 gives 512 bits of scalar capacity.
    ("Viper-128", dict(
        target=128, q=2**12, n=256, k=2,
        eta_s=2, eta_r=2,
        t_pk=9, t_u=9, t_v=4,
        b_msg=1, v_count=1, ss_bits=128,
    )),
    ("Viper-192", dict(
        target=192, q=2**12, n=256, k=3,
        eta_s=3, eta_r=3,
        t_pk=10, t_u=9, t_v=6,
        b_msg=1, v_count=1, ss_bits=192,
    )),
    ("Viper-256", dict(
        target=256, q=2**12, n=256, k=4,
        eta_s=3, eta_r=3,
        t_pk=10, t_u=10, t_v=5,
        b_msg=1, v_count=1, ss_bits=256,
    )),
    ("Viper-384", dict(
        target=384, q=2**13, n=256, k=7,
        eta_s=1, eta_r=1,
        t_pk=11, t_u=11, t_v=5,
        b_msg=2, v_count=1, ss_bits=384,
    )),
    ("Viper-512", dict(
        target=512, q=2**13, n=256, k=9,
        eta_s=1, eta_r=1,
        t_pk=11, t_u=11, t_v=8,
        b_msg=2, v_count=1, ss_bits=512,
    )),
])


SEED_BYTES = 32
HPK_BYTES = 32


def strip_jupyter_kernel_args(argv):
    """
    Jupyter/Sage kernels inject arguments like

        -f /run/user/.../kernel-xxxx.json

    argparse should ignore only these injected kernel arguments.
    Other unknown arguments should still trigger normal argparse errors.
    """
    out = []
    i = 0

    while i < len(argv):
        if argv[i] == "-f" and i + 1 < len(argv):
            nxt = argv[i + 1]
            if "kernel-" in nxt and nxt.endswith(".json"):
                i += 2
                continue

        out.append(argv[i])
        i += 1

    return out


def candidate_estimator_paths(estimator_path):
    """
    Return candidate repository roots containing an estimator/ directory.

    The path passed to lattice-estimator should be the parent directory of
    estimator/, not estimator/ itself.
    """
    candidates = []

    if estimator_path:
        candidates.append(estimator_path)

    candidates += [
        os.environ.get("LATTICE_ESTIMATOR_PATH", ""),
        PROJECT_DIR,
        NOTEBOOK_DIR,
        os.path.abspath(os.getcwd()),
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        os.path.abspath(os.path.join(os.getcwd(), "../..")),
        os.path.abspath(os.path.join(PROJECT_DIR, "lattice-estimator")),
        os.path.abspath(os.path.join(PROJECT_DIR, "..", "lattice-estimator")),
        "/home/ubuntu/桌面/AsiaCrypt/LatticeEstimator",
        "/home/ubuntu/桌面/AsiaCrypt/lattice-estimator",
        "/home/ubuntu/lattice-estimator",
        "/home/ubuntu/Desktop/Lattice estimator/lattice-estimator",
        "/home/dgs/Desktop/Lattice estimator/lattice-estimator",
    ]

    normalized = []
    seen = set()

    for path in candidates:
        if not path:
            continue

        path = os.path.abspath(os.path.expanduser(path))

        if path in seen:
            continue

        seen.add(path)
        normalized.append(path)

    return normalized


def load_estimator(estimator_path):
    tried = []
    import_errors = []

    for path in candidate_estimator_paths(estimator_path):
        tried.append(path)

        if not os.path.isdir(path):
            continue

        candidate_pkg = os.path.join(path, "estimator")

        if not os.path.isdir(candidate_pkg):
            continue

        if path not in sys.path:
            sys.path.insert(0, path)

        try:
            from estimator import LWE, ND
            from estimator.reduction import RC, ADPS16
            print("Loaded lattice-estimator from: %s" % path)
            return LWE, ND, RC, ADPS16
        except Exception as exc:
            import_errors.append((path, repr(exc)))

    msg = []
    msg.append("Could not import lattice-estimator.")
    msg.append("")
    msg.append("The estimator path must be the repository root containing an estimator/ directory.")
    msg.append("")
    msg.append("Your notebook path is:")
    msg.append("  %s" % os.path.join(NOTEBOOK_DIR, "viper_sec_0619.ipynb"))
    msg.append("")
    msg.append("The script first tried:")
    msg.append("  %s" % PROJECT_DIR)
    msg.append("")
    msg.append("Tried paths:")

    for path in tried:
        msg.append("  - %s" % path)

    if import_errors:
        msg.append("")
        msg.append("Import errors from paths containing estimator/:")
        for path, err in import_errors:
            msg.append("  - %s: %s" % (path, err))

    msg.append("")
    msg.append("Run this in a terminal to locate the estimator package:")
    msg.append("  find /home/ubuntu -maxdepth 8 -type d -name estimator 2>/dev/null")
    msg.append("")
    msg.append("Then pass the parent directory, for example:")
    msg.append("  %run viper_mixed_model_estimate.py --estimator-path \"/path/to/parent\"")

    raise RuntimeError("\n".join(msg))


def list_rc_models(RC):
    names = []

    for name in dir(RC):
        if name.startswith("_"):
            continue
        names.append(name)

    return sorted(names)


def make_rc_model(RC, name):
    if not hasattr(RC, name):
        names = ", ".join(list_rc_models(RC))
        raise ValueError(
            "Unknown reduction-cost model RC.%s.\n"
            "Available names:\n%s" % (name, names)
        )

    return getattr(RC, name)


def make_adps_model(ADPS16, mode):
    """
    Compatibility wrapper for different lattice-estimator versions.
    Most versions support ADPS16(mode='classical') and ADPS16(mode='quantum').
    """
    try:
        return ADPS16(mode=mode)
    except TypeError:
        return ADPS16(mode)


def chi_qp_sigma(q, t):
    """
    For q = 2^Q and p = 2^t, the scalar quantization error is

        chi_{q,p} = Uniform{-Delta/2+1, ..., Delta/2},

    where Delta = q/p.

    Its variance is (Delta^2 - 1)/12.
    The lattice-estimator interface is supplied with a variance-matching
    Gaussian surrogate for this bounded distribution.
    """
    p = 2 ** int(t)

    if q % p != 0:
        raise ValueError("p=2^%d does not divide q=%d." % (t, q))

    Delta = q // p

    if Delta == 1:
        return RR(0), Delta, RR(0)

    if Delta % 2 != 0:
        raise ValueError(
            "This power-of-two branch expects even Delta, got Delta=%d." % Delta
        )

    variance = RR(Delta * Delta - 1) / RR(12)
    sigma = sqrt(variance)

    return sigma, Delta, variance


def make_secret_distribution(ND, eta):
    return ND.CenteredBinomial(int(eta))


def make_error_distribution(ND, sigma):
    """
    The true chi_{q,p} distribution is bounded and non-Gaussian.
    We pass a variance-matching Gaussian surrogate to lattice-estimator.
    """
    if sigma == 0:
        try:
            return ND.Uniform(0, 0)
        except Exception:
            pass

    try:
        return ND.DiscreteGaussian(stddev=sigma)
    except TypeError:
        try:
            return ND.DiscreteGaussian(sigma=sigma)
        except TypeError:
            return ND.DiscreteGaussian(sigma)


def make_lwe_params(LWE, ND, profile_name, surface, profile, samples):
    q = int(profile["q"])
    n = int(profile["n"])
    k = int(profile["k"])

    dim = n * k
    m = dim if samples is None else int(samples)

    if surface == "pk":
        eta_secret = int(profile["eta_s"])
        t_error = int(profile["t_pk"])
    elif surface == "u":
        eta_secret = int(profile["eta_r"])
        t_error = int(profile["t_u"])
    else:
        raise ValueError("Unknown surface: %s" % surface)

    sigma, Delta, variance = chi_qp_sigma(q, t_error)

    Xs = make_secret_distribution(ND, eta_secret)
    Xe = make_error_distribution(ND, sigma)

    tag = (
        "%s:%s:dim=%d:m=%d:q=%d:eta=%d:t=%d:Delta=%d:sigma=%.12g"
        % (
            profile_name,
            surface,
            dim,
            m,
            q,
            eta_secret,
            t_error,
            int(Delta),
            float(sigma),
        )
    )

    params = LWE.Parameters(
        n=dim,
        q=q,
        Xs=Xs,
        Xe=Xe,
        m=m,
        tag=tag,
    )

    ss_bits = int(profile["ss_bits"])
    codec_capacity_bits = int(profile["n"]) * int(profile["b_msg"]) * int(profile["v_count"])

    meta = OrderedDict([
        ("profile", profile_name),
        ("target", int(profile["target"])),
        ("surface", surface),
        ("q", q),
        ("n_ring", n),
        ("k", k),
        ("dim", dim),
        ("samples", m),
        ("eta_secret", eta_secret),
        ("t_error", t_error),
        ("Delta", int(Delta)),
        ("sigma_error", float(sigma)),
        ("variance_error", float(variance)),
        ("b_msg", int(profile["b_msg"])),
        ("v_count", int(profile["v_count"])),
        ("fo_bits", ss_bits),
        ("codec_capacity_bits", codec_capacity_bits),
        ("ss_bits", ss_bits),
        ("z_bits", ss_bits),
    ])

    return params, meta


def finite_float(x):
    if x is None:
        return None

    try:
        if x == oo:
            return math.inf
    except Exception:
        pass

    try:
        return float(RR(x))
    except Exception:
        try:
            return float(x)
        except Exception:
            return None


def safe_get(cost, key, default=None):
    keys = [key]

    if key == "beta":
        keys += ["β"]
    if key == "delta":
        keys += ["δ"]
    if key == "zeta":
        keys += ["ζ"]

    for k in keys:
        try:
            return cost[k]
        except Exception:
            pass

        try:
            return getattr(cost, k)
        except Exception:
            pass

    return default


def log2_cost(x):
    """
    Most lattice-estimator cost fields are already log2-costs.

    Some local forks or older versions may store actual operation counts.
    If the finite numeric value is small, treat it as already logarithmic.
    If it is large, take log2.
    """
    y = finite_float(x)

    if y is None:
        return math.inf

    if math.isinf(y):
        return math.inf

    if y < 0:
        return y

    if y < 10000:
        return y

    return float(log(RR(x), 2))


def summarize_result(result):
    rows = []

    for alg, cost in result.items():
        row = OrderedDict()
        row["alg"] = str(alg)
        row["rop_bits"] = log2_cost(safe_get(cost, "rop"))
        row["mem_bits"] = log2_cost(safe_get(cost, "mem"))
        row["red_bits"] = log2_cost(safe_get(cost, "red"))
        row["svp_bits"] = log2_cost(safe_get(cost, "svp"))
        row["guess_bits"] = log2_cost(safe_get(cost, "guess"))
        row["beta"] = finite_float(safe_get(cost, "beta"))
        row["m_used"] = finite_float(safe_get(cost, "m"))
        row["zeta"] = finite_float(safe_get(cost, "zeta"))
        row["raw"] = str(cost)
        rows.append(row)

    finite = [r for r in rows if not math.isinf(r["rop_bits"])]
    best = min(finite, key=lambda r: r["rop_bits"]) if finite else None

    return rows, best


def run_lwe_estimate(LWE, params, red_cost_model, deny_list):
    """
    This function intentionally calls LWE.estimate, not LWE.estimate.rough.

    Therefore the script runs the full estimator routine for the constructed
    LWE anchor, subject only to the explicit deny_list.
    """
    kwargs = {"red_cost_model": red_cost_model}

    if deny_list:
        kwargs["deny_list"] = tuple(deny_list)

    try:
        return LWE.estimate(params, **kwargs)
    except TypeError:
        if deny_list:
            try:
                return LWE.estimate(
                    params,
                    red_cost_model=red_cost_model,
                    deny_list=tuple(deny_list),
                )
            except TypeError:
                pass

        return LWE.estimate(params, red_cost_model=red_cost_model)


def sizes(profile):
    q = int(profile["q"])
    n = int(profile["n"])
    k = int(profile["k"])
    t_pk = int(profile["t_pk"])
    t_u = int(profile["t_u"])
    t_v = int(profile["t_v"])
    v_count = int(profile["v_count"])

    q_bits = int(round(math.log(q, 2)))
    ss_bits = int(profile["ss_bits"])

    ss_bytes = ss_bits / 8
    z_bytes = ss_bytes
    fo_bytes = ss_bytes
    codec_capacity_bytes = n * int(profile["b_msg"]) * v_count / 8

    pk_bytes = SEED_BYTES + k * n * t_pk / 8
    ct_bytes = SEED_BYTES + k * n * t_u / 8 + v_count * n * t_v / 8
    sk_bytes = pk_bytes + HPK_BYTES + z_bytes + k * n * q_bits / 8

    return pk_bytes, ct_bytes, sk_bytes, ss_bytes, fo_bytes, z_bytes, codec_capacity_bytes


def print_header(meta, profile):
    pk_bytes, ct_bytes, sk_bytes, ss_bytes, fo_bytes, z_bytes, codec_capacity_bytes = sizes(profile)

    print("")
    print("=" * 96)
    print("profile          : %s" % meta["profile"])
    print("target           : %d" % meta["target"])
    print("surface          : %s" % meta["surface"])
    print("q, n, k          : %d, %d, %d" % (meta["q"], meta["n_ring"], meta["k"]))
    print("dim, samples     : %d, %d" % (meta["dim"], meta["samples"]))
    print("eta_secret       : %d" % meta["eta_secret"])
    print("t_error          : %d" % meta["t_error"])
    print("Delta            : %d" % meta["Delta"])
    print("sigma_error      : %.12g" % meta["sigma_error"])
    print("b_msg, v_count   : %d, %d" % (meta["b_msg"], meta["v_count"]))
    print("FO message bits  : %d" % meta["fo_bits"])
    print("codec capacity   : %d bits" % meta["codec_capacity_bits"])
    print("shared key bits  : %d" % meta["ss_bits"])
    print("z bits           : %d" % meta["z_bits"])
    print("pk, ct, sk, ss   : %.0f, %.0f, %.0f, %.0f bytes" % (
        pk_bytes,
        ct_bytes,
        sk_bytes,
        ss_bytes,
    ))
    print("FO message bytes : %.0f bytes" % fo_bytes)
    print("z bytes          : %.0f bytes" % z_bytes)
    print("codec capacity   : %.0f bytes" % codec_capacity_bytes)
    print("=" * 96)


def print_result(label, model_name, rows, best):
    print("")
    print("[%s] %s" % (label, model_name))

    for r in sorted(rows, key=lambda x: x["rop_bits"]):
        mem = "inf" if math.isinf(r["mem_bits"]) else "%.3f" % r["mem_bits"]
        red = "inf" if math.isinf(r["red_bits"]) else "%.3f" % r["red_bits"]
        beta = "n/a" if r["beta"] is None else "%.2f" % r["beta"]

        print(
            "  %-18s rop=%9.3f  mem=%9s  red=%9s  beta=%s"
            % (r["alg"], r["rop_bits"], mem, red, beta)
        )

    if best is None:
        print("")
        print("  best_alg  = n/a")
        print("  best_bits = inf")
        return

    print("")
    print("  best_alg  = %s" % best["alg"])
    print("  best_bits = %.6f" % best["rop_bits"])

    if best["beta"] is not None:
        print("  best_beta = %.2f" % best["beta"])


def append_records(records, meta, model_label, model_name, rows):
    for r in rows:
        rec = OrderedDict()
        rec.update(meta)
        rec["model_label"] = model_label
        rec["model_name"] = model_name
        rec.update(r)
        records.append(rec)


def write_csv(path, records):
    fields = [
        "profile",
        "target",
        "surface",
        "model_label",
        "model_name",
        "alg",
        "rop_bits",
        "mem_bits",
        "red_bits",
        "svp_bits",
        "guess_bits",
        "beta",
        "m_used",
        "zeta",
        "q",
        "n_ring",
        "k",
        "dim",
        "samples",
        "eta_secret",
        "t_error",
        "Delta",
        "sigma_error",
        "variance_error",
        "b_msg",
        "v_count",
        "fo_bits",
        "codec_capacity_bits",
        "ss_bits",
        "z_bits",
        "raw",
    ]

    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()

        for r in records:
            writer.writerow({k: r.get(k, "") for k in fields})


def parse_profiles(args):
    profiles = OrderedDict(DEFAULT_PROFILES)

    if args.profile_names.strip().lower() == "all":
        return profiles

    keep = [x.strip() for x in args.profile_names.split(",") if x.strip()]
    selected = OrderedDict()

    for name in keep:
        if name not in profiles:
            raise ValueError(
                "Unknown profile %s. Known profiles: %s"
                % (name, ", ".join(profiles.keys()))
            )

        selected[name] = profiles[name]

    return selected


def print_run_configuration(args, deny_list):
    print("")
    print("Estimator mode: LWE.estimate, not LWE.estimate.rough.")
    print("Length convention: FO message bits = shared key bits, z bits = shared key bits.")
    print("Classical refined model: RC.%s" % args.classical_model)
    print("Classical Core-SVP baseline: ADPS16(mode='classical')")
    print("Quantum Core-SVP model: ADPS16(mode='quantum')")
    print("Denied attacks: %s" % (", ".join(deny_list) if deny_list else "none"))
    print("Profile selection: %s" % args.profile_names)
    print("CSV output: %s" % args.csv)
    print("")


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Viper mixed-model estimator: "
            "classical MATZOV/refined model and quantum Core-SVP."
        )
    )

    parser.add_argument("--estimator-path", default=DEFAULT_ESTIMATOR_PATH)
    parser.add_argument("--classical-model", default="MATZOV")
    parser.add_argument("--deny", default="arora-gb")
    parser.add_argument("--samples", type=int, default=None)
    parser.add_argument("--csv", default="viper_mixed_model_results.csv")
    parser.add_argument("--profile-names", default="all")
    parser.add_argument("--list-models", action="store_true")

    clean_argv = strip_jupyter_kernel_args(sys.argv[1:])
    args = parser.parse_args(clean_argv)

    LWE, ND, RC, ADPS16 = load_estimator(args.estimator_path)

    if args.list_models:
        for name in list_rc_models(RC):
            print(name)
        return

    classical_refined_model = make_rc_model(RC, args.classical_model)
    classical_coresvp_model = make_adps_model(ADPS16, mode="classical")
    quantum_coresvp_model = make_adps_model(ADPS16, mode="quantum")

    deny_list = [x.strip() for x in args.deny.split(",") if x.strip()]

    print_run_configuration(args, deny_list)

    profiles = parse_profiles(args)

    all_records = []
    summary = []

    for profile_name, profile in profiles.items():
        for surface in ["pk", "u"]:
            params, meta = make_lwe_params(
                LWE=LWE,
                ND=ND,
                profile_name=profile_name,
                surface=surface,
                profile=profile,
                samples=args.samples,
            )

            print_header(meta, profile)

            res_cl_refined = run_lwe_estimate(
                LWE,
                params,
                classical_refined_model,
                deny_list,
            )
            rows_cl_refined, best_cl_refined = summarize_result(res_cl_refined)

            print_result(
                "classical refined claim",
                "RC.%s" % args.classical_model,
                rows_cl_refined,
                best_cl_refined,
            )

            res_cl_core = run_lwe_estimate(
                LWE,
                params,
                classical_coresvp_model,
                deny_list,
            )
            rows_cl_core, best_cl_core = summarize_result(res_cl_core)

            print_result(
                "classical Core-SVP baseline",
                "ADPS16(mode='classical')",
                rows_cl_core,
                best_cl_core,
            )

            res_q_core = run_lwe_estimate(
                LWE,
                params,
                quantum_coresvp_model,
                deny_list,
            )
            rows_q_core, best_q_core = summarize_result(res_q_core)

            print_result(
                "quantum Core-SVP claim",
                "ADPS16(mode='quantum')",
                rows_q_core,
                best_q_core,
            )

            append_records(
                all_records,
                meta,
                "classical_refined",
                "RC.%s" % args.classical_model,
                rows_cl_refined,
            )
            append_records(
                all_records,
                meta,
                "classical_coresvp_baseline",
                "ADPS16_classical",
                rows_cl_core,
            )
            append_records(
                all_records,
                meta,
                "quantum_coresvp",
                "ADPS16_quantum",
                rows_q_core,
            )

            summary.append(OrderedDict([
                ("profile", profile_name),
                ("surface", surface),
                ("classical_refined_alg", None if best_cl_refined is None else best_cl_refined["alg"]),
                ("classical_refined_bits", math.inf if best_cl_refined is None else best_cl_refined["rop_bits"]),
                ("classical_refined_beta", None if best_cl_refined is None else best_cl_refined["beta"]),
                ("classical_core_alg", None if best_cl_core is None else best_cl_core["alg"]),
                ("classical_core_bits", math.inf if best_cl_core is None else best_cl_core["rop_bits"]),
                ("classical_core_beta", None if best_cl_core is None else best_cl_core["beta"]),
                ("quantum_core_alg", None if best_q_core is None else best_q_core["alg"]),
                ("quantum_core_bits", math.inf if best_q_core is None else best_q_core["rop_bits"]),
                ("quantum_core_beta", None if best_q_core is None else best_q_core["beta"]),
            ]))

    write_csv(args.csv, all_records)

    print("")
    print("=" * 96)
    print("FINAL BOTTLENECKS")
    print("=" * 96)

    for profile_name in profiles:
        rows = [r for r in summary if r["profile"] == profile_name]

        best_cl_refined = min(rows, key=lambda r: r["classical_refined_bits"])
        best_cl_core = min(rows, key=lambda r: r["classical_core_bits"])
        best_q_core = min(rows, key=lambda r: r["quantum_core_bits"])

        print("")
        print(profile_name)

        print(
            "  classical refined claim     : surface=%s, alg=%s, bits=%.6f, beta=%s"
            % (
                best_cl_refined["surface"],
                best_cl_refined["classical_refined_alg"],
                best_cl_refined["classical_refined_bits"],
                best_cl_refined["classical_refined_beta"],
            )
        )

        print(
            "  classical Core-SVP baseline : surface=%s, alg=%s, bits=%.6f, beta=%s"
            % (
                best_cl_core["surface"],
                best_cl_core["classical_core_alg"],
                best_cl_core["classical_core_bits"],
                best_cl_core["classical_core_beta"],
            )
        )

        print(
            "  quantum Core-SVP claim      : surface=%s, alg=%s, bits=%.6f, beta=%s"
            % (
                best_q_core["surface"],
                best_q_core["quantum_core_alg"],
                best_q_core["quantum_core_bits"],
                best_q_core["quantum_core_beta"],
            )
        )

    print("")
    print("CSV written to: %s" % args.csv)


if __name__ == "__main__":
    main()


Loaded lattice-estimator from: /home/ubuntu/桌面/AsiaCrypt/LatticeEstimator

Estimator mode: LWE.estimate, not LWE.estimate.rough.
Length convention: FO message bits = shared key bits, z bits = shared key bits.
Classical refined model: RC.MATZOV
Classical Core-SVP baseline: ADPS16(mode='classical')
Quantum Core-SVP model: ADPS16(mode='quantum')
Denied attacks: arora-gb
Profile selection: all
CSV output: viper_mixed_model_results.csv


profile          : Viper-128
target           : 128
surface          : pk
q, n, k          : 4096, 256, 2
dim, samples     : 512, 512
eta_secret       : 2
t_error          : 9
Delta            : 8
sigma_error      : 2.29128784748
b_msg, v_count   : 1, 1
FO message bits  : 128
codec capacity   : 256 bits
shared key bits  : 128
z bits           : 128
pk, ct, sk, ss   : 608, 736, 1424, 16 bytes
FO message bytes : 16 bytes
z bytes          : 16 bytes
codec capacity   : 32 bytes
bkw                  :: rop: ≈2^171.1, m: ≈2^158.9, mem: ≈2^159.9, b: 13, t1: 0, t2: